### Lab 2.2: Perceptron Algorithm in PyTorch

In this lab you will again implement the perceptron algorithm, but this time using PyTorch.

In [83]:
import numpy as np
import torch

from palmerpenguins import load_penguins
from mlxtend.plotting import plot_decision_regions
from matplotlib import pyplot as plt

Here we loading and format the Palmer penguins dataset for binary classification.

In [84]:
df = load_penguins()

# drop rows with missing values
df.dropna(inplace=True)

# tricky code to randomly shuffle the rows
df = df.sample(frac=1).reset_index(drop=True)

# select only two specices
df = df[(df['species']=='Adelie')|(df['species']=='Chinstrap')]

# get two features
X = df[['flipper_length_mm','bill_length_mm']].values

# convert speces labels to 0 and 1
y = df['species'].map({'Adelie':0,'Chinstrap':1}).values

To make the learning algorithm work more smoothly, we we will subtract the mean of each feature.

Here `np.mean` calculates a mean, and `axis=0` tells NumPy to calculate the mean over the rows (calculate the mean of each column).

In [85]:
X -= np.mean(X,axis=0)

Now we will convert our `X` and `y` arrays to torch Tensors.

In [86]:
X = torch.tensor(X).float()
y = torch.tensor(y).float()

In [87]:
X

tensor([[-4.9206e+00,  4.1953e+00],
        [ 5.0794e+00,  1.0695e+01],
        [ 3.0794e+00,  7.6953e+00],
        [-1.9206e+00, -6.0047e+00],
        [-8.9206e+00, -1.4047e+00],
        [ 7.9439e-02,  1.1953e+00],
        [ 3.0794e+00,  9.5327e-02],
        [-7.9206e+00, -5.6047e+00],
        [ 7.9439e-02,  4.4953e+00],
        [ 5.0794e+00,  3.7953e+00],
        [ 1.0794e+00,  9.2953e+00],
        [ 1.3079e+01, -9.0467e-01],
        [ 9.0794e+00,  9.9953e+00],
        [-6.9206e+00, -3.0047e+00],
        [-9.2056e-01, -3.0047e+00],
        [-5.9206e+00, -2.4047e+00],
        [ 1.5079e+01,  1.3795e+01],
        [ 4.0794e+00, -2.4047e+00],
        [-1.0921e+01, -3.1047e+00],
        [ 7.0794e+00,  6.0953e+00],
        [ 5.0794e+00,  1.1953e+00],
        [-4.9206e+00, -1.4047e+00],
        [ 2.0794e+00, -7.0467e-01],
        [-5.9206e+00, -3.0047e+00],
        [-1.9206e+00, -3.9047e+00],
        [-9.2056e-01,  2.9533e-01],
        [ 6.0794e+00,  3.1953e+00],
        [-1.9206e+00, -2.304

### Exercises

Your task is to again complete this class for the perceptron, with two changes from last time:
- the implementation should use PyTorch tensors, not NumPy arrays;
- `train_step` now accepts the entire dataset as input and should calculate the average update using all examples at once, rather than updating the weights one data point at a time.

In [88]:
class Perceptron:
    def __init__(self,lr=1e-3):
        # store the learning rate
        self.lr = lr

        # initialize the weights to small, normally-distributed values
        self.w = torch.normal(mean=0, std=0.01, size=(2,))

        # initialize the bias to zero
        self.b = torch.zeros(1)

    def train_step(self,X:torch.Tensor,y:torch.Tensor) -> None:
        """ Apply the least squares update rule shown in lecture.
            This time, the update should be averaged over all data points.
            Arguments:
             X: data matrix of shape (N,2)
             y: labels of shape (N,) 
        """
        # WRITE CODE HERE
        # hint: first convert the y values to -1 or 1 using arithmetic or np.where
        y = torch.where(y > 0, 1.0, -1.0)
        z = X @ self.w + self.b
        errors = y - z
        
        self.w += self.lr * (errors[:, None] * X).mean(dim=0)
        self.b += self.lr * errors.mean()
    
    def predict(self,X:torch.Tensor) -> torch.Tensor:
        """ Calculate model prediction for all data points.
            Arguments:
             X: data matrix of shape (N,2)   
            Returns:
             Predicted labels (0 or 1) of shape (N,)
        """
        # WRITE CODE HERE
        z = X @ self.w + self.b
        return torch.where(z > 0, 1, 0)
    
    def score(self,X:torch.Tensor,y:torch.Tensor) -> torch.Tensor:
        """ Calculate model accuracy
            Arguments:
             X: data matrix of shape (N,2)   
             y: labels of shape (N,)
            Returns:
             Accuracy score
        """
        # WRITE CODE HERE
        y_pred = self.predict(X)
        return (y_pred == y).float().mean()



Run the following code to train the model and print out the accuracy at each step.

In [89]:
lr = 1e-3
epochs = 100
model = Perceptron(lr)
for i in range(epochs):
    model.train_step(X,y)
    print(f'step {i}: {model.score(X,y)}')

step 0: 0.5280373692512512
step 1: 0.7616822719573975
step 2: 0.8878504633903503
step 3: 0.9252336621284485
step 4: 0.9205607771873474
step 5: 0.8971962332725525
step 6: 0.8785046935081482
step 7: 0.8785046935081482
step 8: 0.8738317489624023
step 9: 0.8691588640213013
step 10: 0.8691588640213013
step 11: 0.8644859790802002
step 12: 0.8644859790802002
step 13: 0.8644859790802002
step 14: 0.8644859790802002
step 15: 0.8644859790802002
step 16: 0.8644859790802002
step 17: 0.8644859790802002
step 18: 0.8644859790802002
step 19: 0.8644859790802002
step 20: 0.8644859790802002
step 21: 0.8644859790802002
step 22: 0.8644859790802002
step 23: 0.8644859790802002
step 24: 0.8691588640213013
step 25: 0.8691588640213013
step 26: 0.8691588640213013
step 27: 0.8691588640213013
step 28: 0.8691588640213013
step 29: 0.8785046935081482
step 30: 0.8785046935081482
step 31: 0.8831775784492493
step 32: 0.8831775784492493
step 33: 0.8831775784492493
step 34: 0.8831775784492493
step 35: 0.8831775784492493
st

Run the training multiple times.  Is the training the same each time, or does it vary?  Why?

The training varies because the initial weights are generated randomly. 

Play with the learning rate and number of epochs to find the best setting.

In [103]:
lr = 1e-2
epochs = 150
model = Perceptron(lr)
for i in range(epochs):
    model.train_step(X,y)
    print(f'step {i}: {model.score(X,y)}')

step 0: 0.827102780342102
step 1: 0.8457943797111511
step 2: 0.855140209197998
step 3: 0.8831775784492493
step 4: 0.8831775784492493
step 5: 0.8831775784492493
step 6: 0.9158878326416016
step 7: 0.9205607771873474
step 8: 0.9299065470695496
step 9: 0.9345794320106506
step 10: 0.9299065470695496
step 11: 0.9299065470695496
step 12: 0.9299065470695496
step 13: 0.9345794320106506
step 14: 0.9392523169517517
step 15: 0.9392523169517517
step 16: 0.9392523169517517
step 17: 0.9392523169517517
step 18: 0.9392523169517517
step 19: 0.9392523169517517
step 20: 0.9439252614974976
step 21: 0.9439252614974976
step 22: 0.9439252614974976
step 23: 0.9439252614974976
step 24: 0.9439252614974976
step 25: 0.9439252614974976
step 26: 0.9439252614974976
step 27: 0.9439252614974976
step 28: 0.9485981464385986
step 29: 0.9485981464385986
step 30: 0.9532710313796997
step 31: 0.9532710313796997
step 32: 0.9532710313796997
step 33: 0.9532710313796997
step 34: 0.9532710313796997
step 35: 0.9532710313796997
step

The best one I found was a learning rate of 0.001 and 150 epochs. Raising the number of epochs to 200 didn't really make a difference, so the model needed around 150 to converge with a learning rate of 0.001. 